# COVID-19, bakteriális és vírusos pneumonia elkülönítése mellkasröntgen-felvételek alapján

Neurális hálózat alapú tüdőszegmentáció és négyosztályos képosztályozás (TensorFlow / Keras környezetben).

**Szerző:** Ambrus Csaba



## 0. Környezet és függőségek

A tároló klónozása (vagy frissítése), a `requirements.txt` alapján telepített Python-csomagok, valamint a TensorFlow futásidejű beállítása (GPU, memória) történik itt. Colab esetén a Google Drive csatolása és a Kaggle API-kulcs elhelyezése is ehhez a lépéshez tartozik.


In [ ]:
from pathlib import Path
import os

BRANCH = "main"
REPO_URL = "https://github.com/csambrus/CXR.git"
REPO_DIR = "/content/CXR"

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

%cd /content
if not os.path.exists(f"{REPO_DIR}/.git"):
    !git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
%pip install -r requirements.txt

!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

from src.runtime import setup_tensorflow_runtime

setup_tensorflow_runtime()


# 1. Adatok betöltése

A klasszifikációs és a szegmentációs feladathoz szükséges nyers adatállományok a Kaggle forrásokból kerülnek letöltésre; a szkript opcionálisan helyi zip-archívumot és Google Drive cache-t is használhat (hosszabb futásoknál időt takarít meg). A fejezet végén a várt könyvtárstruktúra áll rendelkezésre a további előfeldolgozáshoz.


In [ ]:
from pathlib import Path
from src.download_dataset import download_all_datasets

from src.config import (
    SEGMENTATION_RAW_DIR,
    SEGMENTATION_DATA_DIR,
)

# True: törli a .dataset_ready markereket és (Colabon) a Drive-os zip + régi mappa-cache-t is,
# majd a Kaggle-ről tiszta lappal tölt újra. Normál futáshoz állítsd False-ra.
FORCE_FRESH_DATASET_DOWNLOAD = True

print("SEGMENTATION_RAW_DIR:", SEGMENTATION_RAW_DIR)
print("SEGMENTATION_DATA_DIR:", SEGMENTATION_DATA_DIR)

download_all_datasets(force=FORCE_FRESH_DATASET_DOWNLOAD)


# 2. Szegmentáció (tüdő kontúrja)

## Cél és kapcsolat a klasszifikációval

A mellkasröntgenen tanított **négyosztályos** osztályozók gyakran olyan képi régiókra is „odanéznek”, amelyek klinikailag kevésbé relevánsak (pl. domborzat, extrapulmonális háttér). A **Grad-CAM** típusú magyarázók ezt vizuálisan is kimutathatják. Ezért a 3. fejezetben ugyanazt a klasszifikációs feladatot **három bemeneti reprezentáción** vizsgáljuk: **nyers** kép, a tüdőterületre **maszkolt** kép, illetve a tüdő körül **vágott (crop)** kép.

Ehhez előbb egy **U-Net jellegű** encoder–decoder hálózatot tanítunk: bemenete a szegmentációs adatbázis párosított röntgen–maszk képe, kimenete pixelenkénti tüdő-valószínűség (bináris maszk). Tanításkor tipikusan **BCE + Dice** jellegű összetett veszteséget, értékeléskor pedig többek között **Dice**-t és **IoU**-t használunk; a tanulás alakját egy összefoglaló görbeábrán követjük.


## 2.1 Képek előkészítése, ellenőrzése

A Kaggle-ről érkező szegmentációs forrás (`segment_raw`) alapján összeállítjuk az **egységes tanító könyvtárat**: párosított röntgen- és referenciamaszk-fájlok, közös névkonvencióval. A cella **PNG-integritást** is ellenőriz (sérült vagy üres fájlok kiszűrése), hogy a `tf.data` pipeline és a későbbi variánsgenerálás ne hibás pixeltől függjön.

Ha már egyszer lefutott az előkészítés, a szkript tipikusan a **meglévő** állományokat hagyja érintetlenül; csak szükség esetén kell újraépíteni a köztes mappákat.


In [ ]:
from src.lung_segmentation import prepare_segmentation_dataset, verify_png_files

verify_png_files(SEGMENTATION_RAW_DIR / "images", "Images")
verify_png_files(SEGMENTATION_RAW_DIR / "masks", "Masks")
prepare_segmentation_dataset(overwrite = False)

## 2.2 Osztás, tanítás, kiértékelés

**Adatbontás:** a szegmentációs azonosítók **tanító / validációs / teszt** részhalmazokba kerülnek (CSV-kben rögzítve), tipikusan **70% / 15% / 15%** arányban. A tanító adaton **adatnövelés** (pl. vízszintes tükrözés, fényesség–kontraszt) is alkalmazható; a validációs és teszt folyam determinisztikus marad.

**Tanítás:** U-Net, Adam optimalizáló, **callback-ek** (legjobb modell mentése a validációs Dice szerint, korai leállítás, tanulási ráta-csökkentés, epochonkénti napló). Ha ugyanazzal a split-fingerprinttel és hiperparaméterekkel már létezik lefutott modell, a kód **kihagyhatja** az újratanítást.

**Kiértékelés:** a teszt halmazon **veszteség** és szegmentációs **metrikák** (Dice, IoU). A tanulási görbe egy ábrán, **három panelen** (veszteség, Dice, IoU) jelenik meg, hogy a túltanulás és a validációs viselkedés egy pillantással áttekinthető legyen.


In [ ]:
from src.lung_segmentation import create_splits, train_segmentation, evaluate_segmentation, plot_training_history

create_splits()
train_segmentation(epochs=25)
evaluate_segmentation()
plot_training_history()

## 2.3 Predikciós minták (kvalitatív ellenőrzés)

A teszt halmazból **néhány** példát ábrázolunk: eredeti kép, referenciamaszk, illetve a modell **0,5 küszöb** mellett binárisított predikciója. Alapértelmezés szerint a minták **véletlenszerűen** kerülnek kiválasztásra, a `src.config.SEED` értékével **reprodukálható** módon (`plot_predictions`, `sample_order="random"`). Ugyanazok sorrend szerinti megjelenítéséhez a `sample_order="sequential"` paraméter használható.

Ez a lépés nem helyettesíti a számszerű tesztmetrikákat, de jól mutatja a tipikus hibatípusokat (alul- vagy túlszegmentálás, élek menti zaj).


In [ ]:
from src.lung_segmentation import plot_predictions

plot_predictions()

## 2.4 Variánsok a klasszifikációhoz: nyers, crop, maszkolt

A szegmentációs modell legjobb súlyaival a **klasszifikációs nyers képeken** (azonos fájlazonosítók) legyártjuk a **tüdőre maszkolt** és a **tüdő körül vágott** változatot. Így a 3. fejezetben mindhárom bemenet **ugyanazt a címkézést** használja; az esetleges teljesítménykülönbség elsősorban a **térbeli tartalom** (háttér vs. tüdődominancia) hatásának tulajdonítható.

A `generate_classifier_variants()` futása **összefoglaló statisztikát** (feldolgozott képek, hibák) és opcionálisan **ábrát** is ment. A variánsok könyvtárai a konfigurációban rögzített útvonalakon érhetők el (`raw`, `lung_crop`, `lung_masked`).


In [ ]:
from pathlib import Path

from IPython.display import Image, display

from src.lung_segmentation import generate_classifier_variants

variant_summary = generate_classifier_variants()
fig_path = variant_summary.get("summary_figure_path")
if fig_path and Path(fig_path).exists():
    display(Image(filename=fig_path))


# 3. Klasszifikáció

## Feladat és kísérleti terv

**Osztályok (négyes):** tipikusan *Normal*, *COVID-19*, *vírusos pneumonia*, *bakteriális pneumonia* — a pontos mappa–címke leképezés a `CLASS_INFOS` táblában van rögzítve.

**Bemenetek:** ugyanazon beteg-/felvétel-azonosítók mellett **három képkészlet** (`raw`, `lung_crop`, `lung_masked`), hogy összehasonlítható legyen a tiszta háttér vs. maszkolt tüdő hatása.

**Modellek:** egy kisebb, **nem előtanított** `baseline_cnn`, valamint három **ImageNet előtanítású** gerinc (**EfficientNetB0**, **ResNet50**, **VGG16**). Transfer esetben kétfázisú stratégia: előbb **zárolt gerinccel** tanított osztályozófej, majd — ahol engedélyezett — **alacsony tanulási rátájú finomhangolás**.

**Metrikák:** a notebook kimenetei között szerepel többek között **pontosság**, **makró átlagolt recall / F1**, **makró one-vs-rest ROC-AUC**; a makróértékek **minden osztályt azonos súlygal** átlagolnak, ami enyhén egyensúlytalan adatnál informatívabb lehet, mint a puszta pontosság.

A fejezet lépései: split ellenőrzés → előtanított súlyok előkészítése → futási konfigurációk → tanítási pipeline → eredmények összefűzése és rangsor → ábrák → magyarázhatóság → legjobb futások kijelölése.


In [ ]:
import pandas as pd
from pathlib import Path

from src.config import (
    RAW_DIR,
    LUNG_MASKED_DIR,
    LUNG_CROP_DIR,
    MODELS_DIR,
    SPLITS_DIR,
    OUTPUT_DIR,
    ensure_dir,
)
from src.dataloader import print_split_summary
from src.compare_explainability import run_compare_explainability
from src.pipeline import (
    get_default_model_run_configs,
    run_training_pipeline,
    build_final_comparison,
    generate_final_plots,
    report_best_models,
    print_pipeline_leaderboard,
    save_pipeline_config,
)

RUN_FULL_RETRAIN = True

DATA_VARIANTS = ["raw", "lung_crop", "lung_masked"]  # raw, crop, masked
FINAL_COMPARISON_NAME = "project_final_optimized"
FINAL_OUT_DIR = ensure_dir(Path(MODELS_DIR) / FINAL_COMPARISON_NAME)

print("RAW_DIR        :", RAW_DIR)
print("LUNG_MASKED_DIR:", LUNG_MASKED_DIR)
print("LUNG_CROP_DIR  :", LUNG_CROP_DIR)
print("MODELS_DIR     :", MODELS_DIR)
print("SPLITS_DIR     :", SPLITS_DIR)
print("OUTPUT_DIR     :", OUTPUT_DIR)
print("FINAL_OUT_DIR  :", FINAL_OUT_DIR)


## 3.1 Adathalmaz megosztása: tanítás / validáció / teszt (70% / 15% / 15%)

**Stratifikált** véletlen bontást alkalmazunk: az egyes kórképek aránya a tanító, validációs és teszt részhalmazokban közel azonos marad. Ez csökkenti annak esélyét, hogy valamelyik ritkább osztály egy részhalmazból „kihulljon”.

**Variáns-konzisztencia:** ellenőrizzük, hogy a **nyers**, a **maszkolt** és a **crop** képtár **ugyanazon** azonosítók szerint épül fel. Így egy adott modell különböző bemenetei **ugyanazt a tesztet** kapják; az összehasonlítás nem szenved el olyan torzítástól, mintha a három variánshoz külön, független véletlen split tartozna.


In [ ]:
from src.dataloader import create_splits, print_split_summary
from src.config import RAW_DIR, SPLITS_DIR
from pathlib import Path

# Convert RAW_DIR and SPLITS_DIR to Path objects if they are strings
source_root_path = Path(RAW_DIR)
split_dir_path = Path(SPLITS_DIR)

splits = create_splits(
    source_root=source_root_path,
    split_dir=split_dir_path,
    overwrite=True,
)

print_split_summary(split_dir_path)


## 3.2 Előtanított gerincek: ImageNet-súlyok letöltése és modellépítés ellenőrzése

Az **EfficientNetB0**, a **ResNet50** és a **VGG16** ImageNeten előtanított súlyai a `train.build_model(..., pretrained=True)` híváskor töltődnek a **Keras modell-cache**-be (tipikusan `~/.keras/models/` alatt). A szürkeárnyalatos röntgenből a hálózat felé **három csatornás** bemenetet állítunk elő (szürkeárnyalat ismétlése), mert az ImageNet gerincek három csatornás képre épültek.

Ha ezt a cellát kihagynánk, az első hosszabb tanítás közben jelentkezne a súlyok letöltése is, ami felesleges várakozást okozhat. A cella **mindhárom** transfer-architektúrát egyszer felépíti, és ellenőrzi, hogy a bemeneti láncban megvan a **`scale_to_255`** réteg (a [0,1] tartományból az ImageNet-preprocesszálás által elvárt skálára).

A **`baseline_cnn`** nem ImageNet-előtanítású: súlyait a 3.4-es tanítás inicializálja; ehhez a cellához nincs szükség külön letöltésre.


In [ ]:
from src.train import build_model

# Mindhárom transfer modell: ImageNet súlyok első letöltése ide esik (ha még nincs cache).
_TRANSFER_MODELS = ("efficientnetb0", "resnet50", "vgg16")

for model_name in _TRANSFER_MODELS:
    print("\n" + "=" * 72)
    print("BUILD + weights:", model_name)
    print("=" * 72)
    _m, _base = build_model(model_name, pretrained=True)
    layer_names = [layer.name for layer in _m.layers]
    print("Első rétegek:", layer_names[:12], "...")

    if "scale_to_255" not in layer_names:
        raise RuntimeError(
            f"[ERROR] {model_name}: nincs `scale_to_255` réteg (train.build_transfer_model). "
            "Ellenőrizd a train.py-t."
        )
    print(f"[OK] {model_name}: scale_to_255 megvan, ImageNet súlyok betölthetők.")
    del _m, _base

print("\n[OK] Mindhárom előre tanított (transfer) modell előkészítve a training előtt.")


## 3.3 Modellfutási konfigurációk

A `get_default_model_run_configs(...)` által visszaadott struktúrák írják le, hogy **mely modellekre**, **mely adatvariánsokra**, **hány epochra** és **milyen tanulási rátával** fusson a tanítás, illetve hogy engedélyezett-e a **finomhangolás**. A táblázatos kimenet a dolgozat **mellékleteként** is archiválható; a `save_pipeline_config(...)` a kiválasztott könyvtárba menti a futtatási paramétereket is, így a kísérletek **reprodukálhatók** maradnak.

A notebookban a `DATA_VARIANTS` és a `RUN_FULL_RETRAIN` változók együtt határozzák meg a **variánsok körét** és azt, hogy minden modellt **nulláról** tanítunk-e, vagy a már kész kimeneteket **újra felhasználjuk**.


In [ ]:
MODEL_RUN_CONFIGS = get_default_model_run_configs(DATA_VARIANTS)

save_pipeline_config(
    run_configs=MODEL_RUN_CONFIGS,
    final_out_dir=FINAL_OUT_DIR,
    run_full_retrain=RUN_FULL_RETRAIN,
    data_variants=DATA_VARIANTS,
)

pd.DataFrame([cfg.__dict__ for cfg in MODEL_RUN_CONFIGS])


## 3.4 Tanítási pipeline futtatása

A `run_training_pipeline` sorrendben végigviszi a 3.3-ban rögzített konfigurációkat. Minden modell–adatvariáns párosra külön könyvtárban keletkeznek a **legjobb** és **utolsó** súlyok, a **tanítási előzmény CSV**, illetve a tanulási görbe **PNG** állománya. A pipeline végén egy **egyesített eredménytábla** (`DataFrame`) áll össze a tesztmetrikákkal.

A `RUN_FULL_RETRAIN=False` beállítással a már sikeresen befejezett futások **kihagyhatók**; így a 3.5–3.8 lépésekhez nem kell minden alkalommal újratanítani (különösen Colab időkorlát mellett hasznos). Első teljes kísérlethez állítsuk `True`-ra, majd értékelési / ábrázási ismétléshez tipikusan `False`-ra.


In [ ]:
model_results_df = run_training_pipeline(
    split_dir=SPLITS_DIR,
    run_configs=MODEL_RUN_CONFIGS,
    run_full_retrain=RUN_FULL_RETRAIN,
    models_dir=MODELS_DIR,
)

print("[OK] Pipeline training lepes kesz.")
print("\n[CHECK] model x variant:")
print(model_results_df.groupby(["model", "data_variant"]).size())

print("\n[CHECK] models:")
print(model_results_df["model"].unique())

print("\n[CHECK] variants:")
print(model_results_df["data_variant"].unique())


## 3.5 Végső összefűzés és ranglista

A külön futásokból egyetlen **`comparison_df`** tábla épül fel. A **`model`** és **`data_variant`** mezőkből képzett kulcs szerinti **deduplikáció** biztosítja, hogy minden (architektúra × bemenet) kombináció egyszer szerepeljen; így a rangsorolás és a további ábrák **egy konzisztens táblára** támaszkodnak.

A blokk a ranglistához szükséges **CSV/JSON** (vagy projektspecifikus összefoglaló) fájlokat is elmenti, hogy a beadandóban hivatkozható legyen a pontos futás és időbélyeg nélküli **számszerű összevetés**.


In [ ]:
comparison_df, FINAL_OUT_DIR = build_final_comparison(
    model_results_df=model_results_df,
    final_comparison_name=FINAL_COMPARISON_NAME,
    models_dir=MODELS_DIR,
)

print_pipeline_leaderboard(comparison_df)
display(comparison_df)


## 3.6 Végső vizualizációs pipeline

A `comparison_df` alapján készülnek az **összehasonlító ábrák**: összesített metrikadiagramok (pl. pontosság, makró-F1, ROC-AUC), tanulási görbék, valamint modell- és variánsszintű áttekintések. Ezek a mellékletek a dolgozat **szöveges eredményeit** egészítik ki; érdemes őket a fejezet megfelelő alfejezeteihez hivatkozással párosítani.

A generált fájlok a `FINAL_OUT_DIR` alatti mappában (vagy a pipeline által kinyomtatott útvonalakon) találhatók — a pontos elérési út a futás naplójában jelenik meg.


In [ ]:
generate_final_plots(comparison_df, FINAL_OUT_DIR, show=True)
print("[OK] Final comparison plots saved to:", FINAL_OUT_DIR)


## 3.7 Magyarázhatóság (explainability)

A **Grad-CAM** (klassz aktivációs térképek) és a **gradiens alapú szaliencia** együttes használata azt segíti megítélni, **mely képi régiók** befolyásolják legjobban az adott osztály predikcióját. Különösen érdekes az összevetés a **nyers**, **maszkolt** és **crop** bemenetek között: ha a maszkolás után jelentősen változik a figyelem eloszlása, az erősíti a 2. fejezetben vázolt motivációt.

**Fontos megjegyzés:** ezek az eszközök **nem klinikai diagnosztika**; a vizualizációk a modell viselkedésének magyarázatára szolgálnak, nem helyettesítik az orvosi értékelést.


In [ ]:
explainability_summary = run_compare_explainability(
    model_names=["baseline_cnn", "efficientnetb0", "resnet50", "vgg16"],
    data_variants=DATA_VARIANTS,
    split_dir=SPLITS_DIR,
    out_dir=Path(OUTPUT_DIR) / "figures" / "project_final_explainability",
    n_examples=6,
    include_saliency=True,
    show=True,
    skip_existing=False,
)
explainability_summary


## 3.8 Legjobb modell kiválasztása

A blokk modellenként és adatvariánsonként kijelöli a **makró-F1** szerinti legjobb futást. A makró-F1 az osztályokra külön számolt F1-értékek **számtani közepe** (egyenlő súlyozás), ezért enyhén egyensúlytalan adatnál informatívabb lehet, mint a puszta pontosság.

Az így kapott táblázat a dolgozat **„legjobb eredmény”** táblázataként is szolgálhat, és alapot ad a további szöveges értékeléshez vagy — ha a kurzus megköveteli — egy egyszerűsített **„üzembe helyezési”** prioritási sorrendhez.


In [ ]:
from pathlib import Path

from IPython.display import Image, display

best_by_variant, best_by_model = report_best_models(comparison_df, FINAL_OUT_DIR, show_plots=True)

print("Best by variant:")
display(best_by_variant)

print("Best by model:")
display(best_by_model)

for rel in ("best_models_plots/best_by_variant_metrics_row.png", "best_models_plots/best_by_model_metrics_row.png"):
    p = FINAL_OUT_DIR / rel
    if p.exists():
        display(Image(filename=str(p)))


# 4. Zárás

Google Colab környezetben érdemes elmenteni a notebookot (és a generált kimeneteket a Drive-on vagy a tárolóban), hogy a hosszú futások eredményei ne vesszenek el a munkamenet lezárásakor.



In [ ]:
from pathlib import Path
import shutil

from src.config import IS_COLAB, OUTPUT_DIR, LOGS_DIR, GDRIVE_OUTPUT, ensure_dir

if IS_COLAB:
    gdrive_out = ensure_dir(GDRIVE_OUTPUT)
    output_src = Path(OUTPUT_DIR)
    logs_src = Path(LOGS_DIR)

    output_dst = gdrive_out / "outputs"
    logs_dst = gdrive_out / "logs"

    if output_src.exists():
        shutil.copytree(output_src, output_dst, dirs_exist_ok=True)
        print("[OK] OUTPUT_DIR copied to:", output_dst)
    else:
        print("[WARN] OUTPUT_DIR does not exist:", output_src)

    if logs_src.exists():
        shutil.copytree(logs_src, logs_dst, dirs_exist_ok=True)
        print("[OK] LOGS_DIR copied to:", logs_dst)
    else:
        print("[WARN] LOGS_DIR does not exist:", logs_src)
else:
    print("[INFO] Not running in Colab, skip Drive sync.")

try:
    from google.colab import _message
    _message.blocking_request("request_save", timeout_sec=10)
    print("Notebook save requested.")
except Exception as e:
    print("[WARN] Notebook save request failed or not running in Colab:", e)
